# Ticket 4 - Recuperacion RAG en Colab

Este notebook valida el **Ticket 4** del proyecto **BlueSea AI Assistant**.

Objetivo:

- instalar el proyecto en Google Colab;
- confirmar que existan documentos, chunks e indice vectorial;
- regenerar `chunks.jsonl` e `vectors.jsonl` si hace falta;
- probar la recuperacion hibrida con fuentes `[S1]`, `[S2]`, etc.;
- validar las consultas criticas de vacaciones, onboarding y beneficios.

La logica final vive en `src/rag_bsf/`. Este notebook solo ejecuta el pipeline como evidencia reproducible.

## 1. Clonar el repositorio

Ejecuta esta celda en Google Colab. Si ya estas dentro del repositorio, la celda solo confirma la ruta.

In [ ]:
from pathlib import Path
import os

repo_dir = Path("bluesea-ai-assistant")

if not Path("pyproject.toml").exists():
    if not repo_dir.exists():
        !git clone https://github.com/aantoa/bluesea-ai-assistant.git
    os.chdir(repo_dir)

print("Ruta actual:", Path.cwd())

## 2. Instalar dependencias

`requirements.txt` incluye `-e .`, por eso Colab instala el paquete local desde `src/rag_bsf/` y permite ejecutar `python -m rag_bsf.cli`.

In [ ]:
!python --version
!pip install -r requirements.txt

## 3. Preparar rutas del proyecto

Esta celda evita errores cuando Colab queda ubicado en otra carpeta.

In [1]:
from pathlib import Path
import os
import sys

# Si sabes la ruta real del proyecto, puedes ponerla aquí.
# Si no, déjalo vacío.
PROJECT_ROOT_OVERRIDE = ""

current = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE
    else Path.cwd().resolve()
)

candidates = [
    current,
    current / "repo",
    current.parent,
    current.parent / "repo",
]

project_root = None

for candidate in candidates:
    if (candidate / "pyproject.toml").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise FileNotFoundError(
        f"No se encontro pyproject.toml desde: {current}. "
        "Revisa si estas en la carpeta correcta o si el proyecto esta dentro de repo/."
    )

if ".Trash" in project_root.parts:
    raise RuntimeError(
        f"Estas usando una copia en la papelera: {project_root}. "
        "Abre la carpeta real del proyecto."
    )

os.chdir(project_root)

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

os.environ["PYTHONPATH"] = str(src_path) + os.pathsep + os.environ.get("PYTHONPATH", "")

chunks_path = project_root / "data" / "processed" / "chunks.jsonl"
manifest_path = project_root / "data" / "index" / "embeddings_manifest.json"
vectors_path = project_root / "data" / "index" / "vectors.jsonl"
documents_dir = project_root / "documents"

print("PROJECT_ROOT =", project_root)
print("src agregado =", src_path)
print("documents =", documents_dir)
print("chunks_path =", chunks_path)
print("manifest_path =", manifest_path)
print("vectors_path =", vectors_path)

PROJECT_ROOT = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant
src agregado = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/src
documents = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/documents
chunks_path = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/processed/chunks.jsonl
manifest_path = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/index/embeddings_manifest.json
vectors_path = /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/index/vectors.jsonl


## 4. Usar los documentos de la carpeta de trabajo

Este notebook esta pensado para tu caso principal: los documentos ya estan dentro de la carpeta del proyecto.

Ruta esperada por defecto:

```text
documents/
  inventory/BSF-INV-001_Document_Inventory.csv
  hr/
  corporate/
  quality/
  hse/
  ...
```

Si esa carpeta ya existe en la raiz del proyecto, no cambies nada: deja `DOCUMENTS_SOURCE_DIR = None`.

Solo usa `DOCUMENTS_SOURCE_DIR` si los documentos estan en otra carpeta. El notebook copiara esa carpeta hacia `documents/` antes de validar, procesar e indexar.


In [3]:
from pathlib import Path
import shutil
import zipfile

# Caso principal: usar la carpeta documents/ que ya esta dentro del proyecto.
# Cambia este valor solo si tus documentos estan en otra carpeta.
# Ejemplos:
# DOCUMENTS_SOURCE_DIR = None
# DOCUMENTS_SOURCE_DIR = '/content/drive/MyDrive/BlueSea/documents'
# DOCUMENTS_SOURCE_DIR = '/content/documents'
DOCUMENTS_SOURCE_DIR = None

# Opcionales para Colab. Mantener en False si trabajas con la carpeta local del proyecto.
MOUNT_GOOGLE_DRIVE = False
RUN_UPLOAD_ZIP = False

# Por defecto este notebook exige documentos fuente reales en documents/.
# Cambialo a True solo si quieres revisar retrieve usando chunks/index ya generados.
ALLOW_EXISTING_ARTIFACTS_WITHOUT_DOCUMENTS = True

DOCUMENTS_DIR = documents_dir
INVENTORY_RELATIVE_PATH = Path('inventory') / 'BSF-INV-001_Document_Inventory.csv'


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]
    except ModuleNotFoundError:
        return False
    return True


def has_inventory(folder: Path) -> bool:
    return (folder / INVENTORY_RELATIVE_PATH).exists()


def resolve_documents_source(source_dir):
    if source_dir is not None and str(source_dir).strip():
        source_path = Path(source_dir).expanduser().resolve()
        nested_documents = source_path / 'documents'
        if not has_inventory(source_path) and has_inventory(nested_documents):
            return nested_documents
        return source_path

    # Default: usar documents/ dentro de la carpeta del proyecto.
    if has_inventory(DOCUMENTS_DIR):
        return DOCUMENTS_DIR

    # Fallback util si el notebook se ejecuta desde la carpeta que contiene inventory/.
    current_dir = Path.cwd().resolve()
    if has_inventory(current_dir):
        return current_dir
    if has_inventory(current_dir / 'documents'):
        return current_dir / 'documents'

    return DOCUMENTS_DIR


def copy_documents_folder(source_dir: Path, target_dir: Path) -> None:
    source_dir = source_dir.resolve()
    target_dir = target_dir.resolve()

    if source_dir == target_dir:
        print('Usando documents/ de la carpeta de trabajo:', target_dir)
        return

    if not source_dir.exists():
        raise FileNotFoundError(f'No existe la carpeta de documentos: {source_dir}')
    if not source_dir.is_dir():
        raise NotADirectoryError(f'La ruta de documentos no es una carpeta: {source_dir}')

    target_dir.mkdir(parents=True, exist_ok=True)
    for item in source_dir.iterdir():
        destination = target_dir / item.name
        if item.is_dir():
            shutil.copytree(item, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(item, destination)
    print('Documentos copiados desde:', source_dir)
    print('Documentos disponibles en:', target_dir)


if MOUNT_GOOGLE_DRIVE:
    if not running_in_colab():
        raise RuntimeError('MOUNT_GOOGLE_DRIVE=True solo funciona dentro de Google Colab.')
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount('/content/drive')

if RUN_UPLOAD_ZIP:
    if not running_in_colab():
        raise RuntimeError(
            'RUN_UPLOAD_ZIP=True solo funciona en Google Colab. '
            'Si estas en local, copia la carpeta documents/ dentro del proyecto '
            'o usa DOCUMENTS_SOURCE_DIR.'
        )
    from google.colab import files  # type: ignore[import-not-found]

    uploaded = files.upload()
    for filename in uploaded:
        if filename.lower().endswith('.zip'):
            with zipfile.ZipFile(filename) as archive:
                archive.extractall(project_root)
            print('Extraido:', filename)
        else:
            print('Archivo cargado, pero no extraido porque no es ZIP:', filename)

source_dir = resolve_documents_source(DOCUMENTS_SOURCE_DIR)
copy_documents_folder(source_dir, DOCUMENTS_DIR)

source_files = [
    path
    for path in DOCUMENTS_DIR.rglob('*')
    if path.is_file() and path.name != '.gitkeep'
]
inventory_file = DOCUMENTS_DIR / INVENTORY_RELATIVE_PATH

print('Carpeta de trabajo:', project_root)
print('Carpeta usada para documentos:', DOCUMENTS_DIR)
print('Archivos fuente detectados:', len(source_files))
print('Inventario esperado:', inventory_file)
print('Inventario existe:', inventory_file.exists())


if not source_files and not ALLOW_EXISTING_ARTIFACTS_WITHOUT_DOCUMENTS:
    raise FileNotFoundError(
        'La carpeta documents/ de este proyecto esta vacia. '
        f'Carpeta revisada: {DOCUMENTS_DIR}. '
        'Ejecuta el notebook desde la carpeta real donde estan tus documentos '
        'o copia ahi inventory/, hr/, corporate/, quality/, hse/, etc.'
    )
elif source_files and not inventory_file.exists():
    raise FileNotFoundError(
        'No se encontro el inventario maestro. '
        'Revisa que documents/ tenga inventory/BSF-INV-001_Document_Inventory.csv'
    )


Usando documents/ de la carpeta de trabajo: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/documents
Carpeta de trabajo: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant
Carpeta usada para documentos: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/documents
Archivos fuente detectados: 25
Inventario esperado: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/documents/inventory/BSF-INV-001_Document_Inventory.csv
Inventario existe: True


## 5. Revisar estructura disponible

Esta vista confirma que el notebook esta leyendo los documentos desde la carpeta `documents/` de tu proyecto actual. Debe aparecer el inventario y los archivos fuente organizados por categoria.


In [4]:
from pathlib import Path

if DOCUMENTS_DIR.exists():
    files_to_show = sorted(
        path.relative_to(project_root)
        for path in DOCUMENTS_DIR.rglob("*")
        if path.is_file() and path.name != ".gitkeep"
    )
    for path in files_to_show[:80]:
        print(path)
    if len(files_to_show) > 80:
        print(f"... {len(files_to_show) - 80} archivos adicionales")
else:
    print("No existe la carpeta documents/.")


documents/.DS_Store
documents/corporate/BSF-CORP-001_Corporate_Profile.pdf
documents/corporate/BSF-CORP-002_Corporate_Glossary.pdf
documents/corporate/BSF-CORP-003_Corporate_Organization_Chart.pdf
documents/corporate/BSF-CORP-004_Corporate_Knowledge_Map.pdf
documents/corporate/BSF-DOC-STD-001_Corporate_Document_Standard.pdf
documents/hr/BSF-HR-001_Employee_Onboarding_Guide.pdf
documents/hr/BSF-HR-002_Employee_FAQ.md
documents/hr/BSF-HR-003_Leave_and_Benefits_Policy.docx
documents/hse/BSF-HSE-001_Workplace_Safety_Procedure.docx
documents/hse/BSF-HSE-002_Emergency_Response_Guide.pdf
documents/inventory/BSF-INV-001_Document_Inventory.csv
documents/inventory/BSF-INV-002_Document_Ownership_Matrix.xlsx
documents/inventory/BSF-INV-003_Document_Sources_and_Ingestion_Plan.md
documents/inventory/BSF-INV-004_Document_Curation_and_Quality_Criteria.md
documents/inventory/BSF-INV-005_Access_and_Permissions_Policy.md
documents/it/BSF-IT-001_BlueTrack_User_Manual.html
documents/it/BSF-IT-002_Corporate

## 6. Validar documentos del Ticket 1

Si cargaste o copiaste documentos fuente, se valida el inventario contra los archivos reales. Si no hay documentos fuente, la validacion se omite para no bloquear una revision basada en artefactos ya generados.


In [5]:
import subprocess
import sys

if source_files:
    subprocess.run([sys.executable, "-m", "rag_bsf.cli", "validate-documents"], check=True)
else:
    print("Validacion omitida: no hay documentos fuente cargados en documents/.")


Document validation completed: 24/24 source files, 24/24 final files, 4/24 Markdown files, 0 missing.


## 7. Procesar documentos y generar chunks

El Ticket 4 consume chunks generados por el Ticket 2. Esta celda ejecuta el procesamiento para asegurar que `data/processed/chunks.jsonl` este actualizado.

In [6]:
import subprocess
import sys
from rag_bsf.config import CHUNKS_FILE

if source_files:
    subprocess.run([sys.executable, "-m", "rag_bsf.cli", "process"], check=True)
elif CHUNKS_FILE.exists() and CHUNKS_FILE.stat().st_size > 0:
    print("Process omitido: no hay documentos fuente, se usara chunks.jsonl existente.")
else:
    raise RuntimeError("No hay documentos fuente ni chunks existentes. Carga documents/ antes de continuar.")


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong po

Processing completed: 24 documents, 1018 chunks.


## 8. Generar indice vectorial

El retrieve usa `data/index/vectors.jsonl`. Esta celda indexa los chunks disponibles con embeddings locales deterministicos.

In [7]:
import subprocess
import sys
from rag_bsf.config import CHUNKS_FILE, VECTOR_INDEX_FILE

if source_files:
    subprocess.run([sys.executable, "-m", "rag_bsf.cli", "index"], check=True)
elif VECTOR_INDEX_FILE.exists() and VECTOR_INDEX_FILE.stat().st_size > 0:
    print("Index omitido: no hay documentos fuente, se usara vectors.jsonl existente.")
elif CHUNKS_FILE.exists() and CHUNKS_FILE.stat().st_size > 0:
    subprocess.run([sys.executable, "-m", "rag_bsf.cli", "index"], check=True)
else:
    raise RuntimeError("No hay documentos fuente, chunks ni indice. Carga documents/ antes de continuar.")


Indexing completed: 1018 chunks, 1018 vectors, 384 dimensions, model local-hashing-v1.


## 9. Confirmar archivos generados

Antes de recuperar contexto, revisa que existan `chunks.jsonl`, `vectors.jsonl` y el manifest del indice.

In [8]:
from rag_bsf.config import CHUNKS_FILE, VECTOR_INDEX_FILE, EMBEDDINGS_MANIFEST_FILE

for path in [CHUNKS_FILE, VECTOR_INDEX_FILE, EMBEDDINGS_MANIFEST_FILE]:
    print(path)
    print("exists=", path.exists(), "bytes=", path.stat().st_size if path.exists() else 0)
    if path.exists() and path.suffix == ".jsonl":
        print("lines=", sum(1 for _ in path.open(encoding="utf-8")))
    print()

/Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/processed/chunks.jsonl
exists= True bytes= 1053979
lines= 1018

/Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/index/vectors.jsonl
exists= True bytes= 3896289
lines= 1018

/Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/index/embeddings_manifest.json
exists= True bytes= 513



## 10. Probar recuperacion por CLI

Estas son las tres consultas criticas que validan el reranking hibrido del Ticket 4. Se usa `candidate_k=30` porque el indice local por hashing necesita un conjunto amplio de candidatos y luego el reranker textual corrige el orden.

In [9]:
!python -m rag_bsf.cli retrieve "cuantos dias de vacaciones tengo" --top-k 5 --candidate-k 30

Retrieval completed: 48 semantic candidates, 48 after filters, 5 context chunks, filters=none.

[S1] BSF-HR-002 | Employee FAQ | section=8.2 How much annual leave do I have? | category=Human Resources | review_date=N/A
## 8.2 How much annual leave do I have?

Your entitlement depends on applicable law, employment conditions, and company policy. Check BlueHR or contact Human Resources.

---

[S2] BSF-HR-003 | Leave and Benefits Policy | section=Document | category=Human Resources | review_date=N/A
request, legal protections, operational impact, prior leave usage, and HR review.

Examples may include:

Urgent personal matters.

Family emergency.

Relocation or essential administrative appointments.

Personal legal obligations.

Extended travel not covered by annual leave.

Recovery or care needs beyond paid entitlement.

Other exceptional circumstances approved by HR.

Unpaid leave shall be documented before the leave starts whenever possible. HR shall explain the effect on payroll, bene

In [10]:
!python -m rag_bsf.cli retrieve "onboarding obligatorio" --top-k 5 --candidate-k 30

Retrieval completed: 52 semantic candidates, 52 after filters, 5 context chunks, filters=none.

[S1] BSF-HR-002 | Employee FAQ | section=4.4 Is onboarding mandatory? | category=Human Resources | review_date=N/A
## 4.4 Is onboarding mandatory?

Yes. All assigned onboarding activities and mandatory training must be completed within the required timeframe.

---

[S2] BSF-HR-001 | Employee Onboarding Guide | section=Page 5 | category=Human Resources | review_date=N/A
documentation; ● ensure that mandatory training is assigned; ● maintain onboarding and personnel records; ● monitor onboarding performance indicators; ● address escalated onboarding issues.

---

[S3] BSF-HR-002 | Employee FAQ | section=4.8 What happens if I do not complete mandatory onboarding? | category=Human Resources | review_date=N/A
## 4.8 What happens if I do not complete mandatory onboarding?

Your duties, access, or authorization may be restricted until the required training and documentation are completed.

---

---

In [11]:
!python -m rag_bsf.cli retrieve "que beneficios tengo como empleado" --top-k 5 --candidate-k 30

Retrieval completed: 54 semantic candidates, 54 after filters, 5 context chunks, filters=none.

[S1] BSF-HR-002 | Employee FAQ | section=9.1 Where can I find information about employee benefits? | category=Human Resources | review_date=N/A
## 9.1 Where can I find information about employee benefits?

Review BSF-HR-003 — Leave and Benefits Policy or contact Human Resources.

---

[S2] BSF-HR-003 | Leave and Benefits Policy | section=Document | category=Human Resources | review_date=N/A
benefits and paid leave for non-employees shall be governed by the relevant contract, agency arrangement, local law, and applicable commercial terms.

The policy covers:

Annual leave or vacation.

Public holidays and company holidays.

Sick leave and medical absence.

Maternity, paternity, parental, adoption, and family-related leave where applicable.

Bereavement and compassionate leave.

Personal, emergency, and unpaid leave.

Study, training, civic, and approved special leave.

Work-related injury or 

## 11. Validacion programatica de resultados esperados

Esta celda falla con `AssertionError` si el primer resultado deja de apuntar a la seccion correcta.

In [12]:
from rag_bsf.rag_pipeline import retrieve_rag_context

checks = [
    ("cuantos dias de vacaciones tengo", "8.2 How much annual leave do I have?"),
    ("onboarding obligatorio", "4.4 Is onboarding mandatory?"),
    ("que beneficios tengo como empleado", "9.1 Where can I find information about employee benefits?"),
]

for question, expected_section in checks:
    retrieved = retrieve_rag_context(question, top_k=5, candidate_k=30)
    assert retrieved.results, f"Sin resultados para: {question}"
    first = retrieved.results[0].chunk.metadata.get("section", "")
    print(question)
    print("S1 section =", first)
    assert expected_section in first, f"Esperado {expected_section!r}, obtenido {first!r}"
    print("OK")
    print()

cuantos dias de vacaciones tengo
S1 section = 8.2 How much annual leave do I have?
OK

onboarding obligatorio
S1 section = 4.4 Is onboarding mandatory?
OK

que beneficios tengo como empleado
S1 section = 9.1 Where can I find information about employee benefits?
OK



## 12. Recuperacion con filtro de metadata

Los filtros permiten restringir por campos como `category`, `document_code`, `status` o `confidentiality`.

In [13]:
!python -m rag_bsf.cli retrieve "onboarding obligatorio" --top-k 5 --candidate-k 30 --filter category="Human Resources"

Retrieval completed: 52 semantic candidates, 46 after filters, 5 context chunks, filters=category=Human Resources.

[S1] BSF-HR-002 | Employee FAQ | section=4.4 Is onboarding mandatory? | category=Human Resources | review_date=N/A
## 4.4 Is onboarding mandatory?

Yes. All assigned onboarding activities and mandatory training must be completed within the required timeframe.

---

[S2] BSF-HR-001 | Employee Onboarding Guide | section=Page 5 | category=Human Resources | review_date=N/A
documentation; ● ensure that mandatory training is assigned; ● maintain onboarding and personnel records; ● monitor onboarding performance indicators; ● address escalated onboarding issues.

---

[S3] BSF-HR-002 | Employee FAQ | section=4.8 What happens if I do not complete mandatory onboarding? | category=Human Resources | review_date=N/A
## 4.8 What happens if I do not complete mandatory onboarding?

Your duties, access, or authorization may be restricted until the required training and documentation are 

## 13. Ejecutar tests relevantes

Esta celda valida las pruebas unitarias principales del Ticket 4 y las pruebas de validacion documental que soportan la ingesta multiformato.

In [14]:
!PYTHONPATH=src python -m pytest tests/test_retrieval.py tests/test_document_validation.py -q

..........                                                               [100%]
10 passed in 0.11s


## 14. Resultado esperado

El Ticket 4 queda correcto cuando:

- `retrieve "cuantos dias de vacaciones tengo"` devuelve primero `8.2 How much annual leave do I have?`;
- `retrieve "onboarding obligatorio"` devuelve primero `4.4 Is onboarding mandatory?`;
- `retrieve "que beneficios tengo como empleado"` devuelve primero `9.1 Where can I find information about employee benefits?`;
- los tests relevantes pasan.

El comando `retrieve` todavia no redacta una respuesta final conversacional. Solo arma el contexto con fuentes para que el siguiente ticket conecte el LLM.